# 1) ResU-Net for Deblur and Denoise operations
### Using Residual U-Net to address noise and blur issues in images: network training and evaluation
_NOTE: This code was written to run on Google Colab. Any repetitions such as loading files at the beginning of section 6 and 7 are due to being able to re-run the network operation without running the entire code (including training)_

## 1. Setup and Initialization
 List of imports, global variables and folders used

In [ ]:
from google.colab import drive
import os
from datasets import load_from_disk
from pathlib import Path
import sys
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau

import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import shutil

from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
!pip install astra-toolbox


### --- DIRECTORIES --- ###
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/Computational Imaging")
DATASET_DIR = BASE_DIR/"dataset"
RESULT_DIR = BASE_DIR/"end_to_end_output"
LOSS_HISTORY_DIR = RESULT_DIR/"loss_history"
METRICS_DIR = RESULT_DIR/"metrics_data"
WEIGHTS_DIR = RESULT_DIR/"weights"
WEIGHTS_PATH = WEIGHTS_DIR/"resunet.pth"
RECONSTRUCTION_DIR = RESULT_DIR/"reconstruction_examples"
LOCAL_DATASET = Path("/content/dataset_local")

IPPY_CONTAINER = BASE_DIR / "IPPy"

sys.path.append(str(BASE_DIR))


# Loop to fix same-named subfolder problem (IPPy)
for key in list(sys.modules.keys()):
    if "IPPy" in key or "utilities" in key:
        del sys.modules[key]

if str(IPPY_CONTAINER) not in sys.path:
    sys.path.insert(0, str(IPPY_CONTAINER))

from IPPy import utilities
from IPPy.utilities.metrics import PSNR, SSIM


### --- HYPERPARAMETERS --- ###
BATCH_SIZE = 8
EPOCH_NUMBER = 30
LEARNING_RATE = 1e-3
PATIENCE_EARLY_STOPPING = 7

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Loading data from the dataset
Pre-computed degraded inputs ensure a fair, standardized comparison across all models while optimizing computational efficiency. The dataset's extensive semantic variety prevents overfitting, forcing the network to learn generalized reconstruction features. Loading and personalization of dataset

In [ ]:
# Necessary to remove corrupt data due to data upload interruptions
if LOCAL_DATASET.exists() and not (LOCAL_DATASET / "dataset_info.json").exists():
    shutil.rmtree(str(LOCAL_DATASET))

# Dataset caching standard
if not LOCAL_DATASET.exists():
    shutil.copytree(str(DATASET_DIR), str(LOCAL_DATASET))

full_dataset = load_from_disk(str(LOCAL_DATASET))

train_data = full_dataset["train"]
test_data = full_dataset["test"]
validation_data = full_dataset["validation"]

print(f"Train size: {len(train_data)}")
print(f"Test size: {len(test_data)}")
print(f"Validation size: {len(validation_data)}")
#print(f"{train_data.features}")


# Creating a virtual expansion of dataset
class DegradedDataset(Dataset):
    def __init__(self, dataset_subset): # torch.utils.data.Dataset __init__ override
        self.data = dataset_subset # dataset_subset: base arrow dataset containing clean and noisy images.
        self.to_tensor = transforms.ToTensor()
        self.noise_cols = ["y_005", "y_010", "y_050", "y_100"]

    def __len__(self): #  torch.utils.data.Dataset __len__ override
        return len(self.data) * len(self.noise_cols) # Setting virtual dataset size 4 times bigger

    def __getitem__(self, idx):  # torch.utils.data.Dataset __getitem__ override
        img_idx   = idx // len(self.noise_cols) # Entire division -> value updated every 4 iterations, row index (source image index)
        noise_idx = idx % len(self.noise_cols) # Remainder operator to cycle through noise levels

        sample = self.data[img_idx]
        clean   = self.to_tensor(sample["x"].convert("RGB"))
        degraded = self.to_tensor(sample[self.noise_cols[noise_idx]].convert("RGB"))

        return  degraded, clean


train_dataset = DegradedDataset(train_data)
val_dataset = DegradedDataset(validation_data)
test_dataset = DegradedDataset(test_data)

# num_workers=2 delegates data loading and tensor conversion to 2 child processes.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,num_workers=2, drop_last=True) # Shuffle training data at the start of each epoch to ensure randomness and prevent overfitting
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)



## 3. Sanity Check (Execution is optional)
extraction of a batch of images to verify the correct correspondency between ground truth images and degraded ones and to check the correctness of dataloader overall

In [ ]:
data_iter = iter(train_loader)
degraded_batch, clean_batch = next(data_iter)

print(f"Clean image tensor:   {clean_batch.shape}\n")
print(f"Degraded image tensor: {degraded_batch.shape}\n")

fig, axes = plt.subplots(4, 2, figsize=(10, 16))
for i in range(4):
    img_degraded = degraded_batch[i].permute(1, 2, 0).numpy()
    img_clean    = clean_batch[i].permute(1, 2, 0).numpy()

    axes[i, 0].imshow(img_clean)
    axes[i, 0].set_title(f"Index {i} - Ground Truth")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(img_degraded)
    axes[i, 1].set_title(f"Index {i} - Degraded Input")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


## 4. ResU-Net architecture
Network Definition: Implementation of the ResUNet structure, combining the U-Net funnel shape with residual blocks

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)  # kernel_size=3, stride=1, padding=1 to preserve spatial resolution
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU()

        if in_channels != out_channels: # Adaptation of the number of shortcut channels to that of the convolution output, using 1D convolutions
          self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
          self.shortcut = nn.Identity()

    def forward(self, x): # x: input tensor
        shortcut = self.shortcut(x)

        #--------- out = conv1 -> norm -> Relu -> conv2 -> norm

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        #---------

        out += shortcut

        out = self.relu(out)
        return out

class DownBlock(nn.Module): # Downsampling with maxpool before ResidualBlock computation
      def __init__(self, in_channels, out_channels, block_type = ResidualBlock):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.block = block_type(in_channels, out_channels)

      def forward(self, x):
        x = self.pool(x)
        x = self.block(x)
        return x

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, block_type = ResidualBlock):
        super().__init__()

        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2) #kernel_size=2, stride=2 -> Upsampling
        self.block = block_type(out_ch + skip_ch, out_ch) # input channels = output channels + skip connection channels.

    def forward(self, x, skip):
        x = self.up(x)

        if x.shape[-2:] != skip.shape[-2:]: # use interpolation to make the spatial dimension equal in case of mismatch due to downsampling above odd dimensions
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)

        x = torch.cat([skip, x], dim=1)
        return self.block(x)



class ResUNet(nn.Module):
    def __init__(self, in_channels = 3, out_channels = 3, base_channels = 32):
        super(ResUNet, self).__init__()

        self.initial_block = ResidualBlock(in_channels, base_channels)

        self.down_block1 = DownBlock(base_channels, base_channels*2)
        self.down_block2 = DownBlock(base_channels*2, base_channels*4)
        self.down_block3 = DownBlock(base_channels*4, base_channels*8)

        self.bottleneck_block = DownBlock(base_channels*8, base_channels*16)

        self.up_block1 = UpBlock(base_channels*16, base_channels*8, base_channels*8)
        self.up_block2 = UpBlock(base_channels*8, base_channels*4, base_channels*4)
        self.up_block3 = UpBlock(base_channels*4, base_channels*2, base_channels*2)
        self.up_block4 = UpBlock(base_channels*2, base_channels, base_channels)

        self.final_block = nn.Conv2d(base_channels, out_channels, kernel_size=1)


    def forward(self, x):
        x1 = self.initial_block(x)
        x2 = self.down_block1(x1)
        x3 = self.down_block2(x2)
        x4 = self.down_block3(x3)

        out = self.bottleneck_block(x4)

        out = self.up_block1(out, x4)
        out = self.up_block2(out, x3)
        out = self.up_block3(out, x2)
        out = self.up_block4(out, x1)

        out = self.final_block(out)
        out = torch.sigmoid(out) # sigmoid mapping for pixel range normalization [0.0, 1.0]

        return out

In [ ]:

# Check output path existence
path_check = {
    "LOSS_HISTORY_DIR":LOSS_HISTORY_DIR,
    "WEIGHTS_DIR": WEIGHTS_DIR,
    "RECONSTRUCTION_DIR": RECONSTRUCTION_DIR,
    "METRICS_DIR" : METRICS_DIR
}

for n, path in path_check.items():
    if path.exists():
        print(f" OK {n}! ({path.name})")
    else:
        print(f"{n}: NOT FOUND: {path}")
        tutto_ok = False

all_ok = True

## 5. Training Loop
Training loop to process data, calculate loss and update weights.

In [ ]:
def training_loop(model, w_path = WEIGHTS_PATH, num_epochs=EPOCH_NUMBER, learning_rate=LEARNING_RATE, patience_early_stopping=PATIENCE_EARLY_STOPPING):
    # patience_early_stopping = 7 allows the scheduler to run twice before ending training

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate) # AdamW for better perfomance

    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5) # after 3 epochs without improvement, it halves the learning
    loss_fn = nn.MSELoss()

    history = {'train_loss': [], 'val_loss': []}

    best_val_loss = float('inf') # tracking the best validation loss
    epochs_no_improve = 0 # counting epochs with no improvement for early stopping

    for epoch in range(num_epochs):

        model.train()
        train_loss = 0.0
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False)
        for degraded_batch, clean_batch in train_bar:
            degraded_batch, clean_batch = degraded_batch.to(device), clean_batch.to(device)

            prediction = model(degraded_batch)
            loss = loss_fn(prediction, clean_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

            train_bar.set_postfix({'loss': f'{loss.item():.6f}'})

        avg_train_loss = train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for degraded_batch, clean_batch in val_loader:
                degraded_batch, clean_batch = degraded_batch.to(device), clean_batch.to(device)

                prediction = model(degraded_batch)
                loss = loss_fn(prediction, clean_batch)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        history['val_loss'].append(avg_val_loss)


        scheduler.step(avg_val_loss)

        # saving weights with lowest validation loss
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), w_path)
            epochs_no_improve = 0
            tqdm.write(f"Epoch {epoch+1}/{num_epochs} -> Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} [BEST MODEL SAVED]")
        else:
            epochs_no_improve += 1
            tqdm.write(f"Epoch {epoch+1}/{num_epochs} -> Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} [No improvement since{epochs_no_improve} epochs]")


        if epochs_no_improve >= patience_early_stopping:
            tqdm.write(f"\n Early Stopping activated. No improvement on validation loss since {patience_early_stopping} epochs.")
            print("Training finished")
            break


    # recovering best model weights to return
    reloaded_model = type(model)()
    reloaded_model.load_state_dict(torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=True))
    reloaded_model = reloaded_model.to(device).eval()


    history_file_path = LOSS_HISTORY_DIR / f"history.json"


    try:
        with open(history_file_path, 'w') as f:

              json.dump(history, f, indent=4)
        tqdm.write(f"[INFO] History saved in: {history_file_path}")
    except Exception as e:
          tqdm.write(f"[ERROR]: {e}")

    return reloaded_model, history

torch.manual_seed(0) # guarantee total reproducibility of your code: ResUNet always starts from the exact same initial weights and DataLoader always mixes in the exact same order.
ResU_Net, history = training_loop(ResUNet())

## 6. Metric-based evaluation
Evaluating the best network weights based on objective metrics:

*NOTE: If you want to evaluate without training, run all mandatory sections except numer 5.*

In [ ]:
if 'ResU_Net' not in locals() and 'ResU_Net' not in globals(): # Reload Model from drive
    print("[INFO] 'ResU_Net' not found in RAM. Restoring from weights...")
    ResU_Net = ResUNet()
    state_dict = torch.load(WEIGHTS_PATH, map_location=device, weights_only=True)
    ResU_Net.load_state_dict(state_dict)
    ResU_Net = ResU_Net.to(device).eval()
    print("[SUCCESS] Pre-trained weights loaded safely.")
else:
    print("[INFO] 'ResU_Net' is already active in RAM. Ready for plotting.")
    ResU_Net.eval()


if 'history' not in locals() and 'history' not in globals(): # Reload loss history from dive
    if LOSS_HISTORY_DIR.exists():
        try:
            with open(history_file_path, 'r') as f:
              history = json.load(f)
            print(f"[SUCCESS] History uploaded from: {history_file_path}")
        except Exception as e:
            print(f"[ERROR]: {e}")

def evaluate_model(model, test_data, device=device):
    model.eval()
    to_tensor = transforms.ToTensor()
    noise_cols = ["y_005", "y_010", "y_050", "y_100"]

    results = {col: {"psnr": [], "ssim": []} for col in noise_cols}

    print("Starting evaluation...")


    # Disable gradient tracking to optimize memory usage and speed up inference
    with torch.no_grad():
        for i in range(len(test_data)):
            sample = test_data[i]


            clean_eval = to_tensor(sample["x"].convert("RGB")).to(device)
            if clean_eval.dim() == 3:
                clean_eval = clean_eval.unsqueeze(0)

            for col in noise_cols:
                degraded = to_tensor(sample[col].convert("RGB")).to(device)

                # geometry checking: dynamically normalize input tensor to 4D (Batch, Channels, Height, Width)
                if degraded.dim() == 3:
                    input_tensor = degraded.unsqueeze(0)
                elif degraded.dim() == 4:
                    input_tensor = degraded
                elif degraded.dim() == 5:
                    input_tensor = degraded.squeeze(1)

                pred_eval = model(input_tensor).clamp(0, 1)


                # Strip accidental 5D outputs to maintain architectural conformity
                if pred_eval.dim() == 5:
                    pred_eval = pred_eval.squeeze(1)


                # Detach tensors from the computational graph and copy to CPU host memory, mandatory as external IPPy metrics rely on NumPy/CPU
                pred_cpu = pred_eval.detach().cpu()
                clean_cpu = clean_eval.detach().cpu()

                current_psnr = float(PSNR(pred_cpu, clean_cpu))
                current_ssim = float(SSIM(pred_cpu, clean_cpu))

                # Append results to their respective tracking lists inside the dictionary
                results[col]["psnr"].append(current_psnr)
                results[col]["ssim"].append(current_ssim)

    summary = {}
    for col in noise_cols:
        summary[col] = {
            "psnr_mean": np.mean(results[col]["psnr"]),
            "psnr_std":  np.std(results[col]["psnr"]),
            "ssim_mean": np.mean(results[col]["ssim"]),
            "ssim_std":  np.std(results[col]["ssim"]),
        }

    df_summary = pd.DataFrame(summary).T
    return df_summary, results
    summary = {}
    for col in noise_cols:
        summary[col] = {
            "psnr_mean": np.mean(results[col]["psnr"]),
            "psnr_std":  np.std(results[col]["psnr"]),
            "ssim_mean": np.mean(results[col]["ssim"]),
            "ssim_std":  np.std(results[col]["ssim"]),
        }

    df_summary = pd.DataFrame(summary).T
    return df_summary, results

metrics_summary, raw_results = evaluate_model(ResU_Net, test_data)

# Saving on CSV file
METRICS_DIR.mkdir(parents=True, exist_ok=True)
metrics_summary.to_csv(METRICS_DIR / "resunet_metrics_per_noise_level.csv")

### ----- Plot ----- ###

train_loss_hist = history['train_loss']
val_loss_hist = history['val_loss']

train_psnr_hist = [-10 * np.log10(loss) for loss in train_loss_hist]
val_psnr_hist = [-10 * np.log10(loss) for loss in val_loss_hist]
epochs_range = range(1, len(train_loss_hist) + 1)

noise_levels_res = metrics_summary.index
psnr_means_res = metrics_summary['psnr_mean']
psnr_stds_res  = metrics_summary['psnr_std']
ssim_means_res = metrics_summary['ssim_mean']
ssim_stds_res  = metrics_summary['ssim_std']

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
ax1, ax2, ax3, ax4 = axes[0,0], axes[0,1], axes[1,0], axes[1,1]

# [0,0] Loss curve
ax1.plot(epochs_range, train_loss_hist, label='Train Loss', color='#1f77b4', linewidth=2)
ax1.plot(epochs_range, val_loss_hist,   label='Val Loss',   color='#ff7f0e', linewidth=2, linestyle='--')
ax1.set_title('Loss Curve (MSE)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epochs', fontsize=12)
ax1.set_ylabel('Mean Squared Error', fontsize=12)
ax1.legend(fontsize=11)
ax1.grid(True, linestyle=':', alpha=0.6)

# [0,1] PSNR curve per epoca
ax2.plot(epochs_range, train_psnr_hist, label='Train PSNR', color='#2ca02c', linewidth=2)
ax2.plot(epochs_range, val_psnr_hist,   label='Val PSNR',   color='#d62728', linewidth=2, linestyle='--')
ax2.set_title('PSNR per Epoca', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epochs', fontsize=12)
ax2.set_ylabel('PSNR (dB)', fontsize=12)
ax2.legend(fontsize=11)
ax2.grid(True, linestyle=':', alpha=0.6)

# [1,0] PSNR bar per noise level
ax3.bar(noise_levels_res, psnr_means_res, yerr=psnr_stds_res, capsize=5,
        color='#2ca02c', edgecolor='black', alpha=0.8)
ax3.set_title('Test Set: PSNR per Noise Level', fontsize=14, fontweight='bold')
ax3.set_xlabel('Noise Column', fontsize=12)
ax3.set_ylabel('PSNR (dB)', fontsize=12)
ax3.grid(True, linestyle=':', alpha=0.6)

# [1,1] SSIM bar per noise level
ax4.bar(noise_levels_res, ssim_means_res, yerr=ssim_stds_res, capsize=5,
        color='#9467bd', edgecolor='black', alpha=0.8)
ax4.set_title('Test Set: SSIM per Noise Level', fontsize=14, fontweight='bold')
ax4.set_xlabel('Noise Column', fontsize=12)
ax4.set_ylabel('SSIM Index (0 to 1)', fontsize=12)
ax4.set_ylim(0, 1.1)
ax4.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.savefig(METRICS_DIR / "resunet_learning_curves.png", dpi=300)
plt.show()

## 7. Visual evaluation
*NOTE: If you want to evaluate without training, run all mandatory sections except number 5 and 6.*

In [ ]:
if 'ResU_Net' not in locals() and 'ResU_Net' not in globals():
    print("[INFO] 'ResU_Net' not found in RAM. Restoring from weights...")
    ResU_Net = ResUNet()
    state_dict = torch.load(WEIGHTS_PATH, map_location=device, weights_only=True)
    ResU_Net.load_state_dict(state_dict)
    ResU_Net = ResU_Net.to(device).eval()
    print("[SUCCESS] Pre-trained weights loaded safely.")
else:
    print("[INFO] 'ResU_Net' is already active in RAM. Ready for plotting.")
    ResU_Net.eval()

# Activation evaluation mode
ResU_Net.eval()

# 2. Random sample from test dataset
random_idx = random.randint(0, len(test_data) - 1)
sample = test_data[random_idx]

to_tensor = transforms.ToTensor()
clean_tensor = to_tensor(sample["x"].convert("RGB"))
img_clean = clean_tensor.permute(1, 2, 0).numpy()

noise_cols = ["y_005", "y_010", "y_050", "y_100"]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx, col in enumerate(noise_cols):
    degraded_tensor = to_tensor(sample[col].convert("RGB"))

    input_batch = degraded_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        output_batch = ResU_Net(input_batch)
        output_tensor = output_batch.squeeze(0).cpu().clamp(0, 1)

    img_degraded = degraded_tensor.permute(1, 2, 0).numpy()
    img_predicted = output_tensor.permute(1, 2, 0).numpy()

    axes[0, idx].imshow(img_degraded)
    axes[0, idx].set_title(f"Input {col}", fontsize=14, fontweight='bold')
    axes[0, idx].axis("off")

    axes[1, idx].imshow(img_predicted)
    axes[1, idx].set_title(f"Output per {col}", fontsize=14, fontweight='bold', color='green')
    axes[1, idx].axis("off")

plt.tight_layout()
plt.savefig(RECONSTRUCTION_DIR / f"example_{random_idx}_grid.png", dpi=300)
plt.show()

plt.figure(figsize=(5, 5))
plt.imshow(img_clean)
plt.title("Ground Truth", fontsize=14, fontweight='bold')
plt.axis("off")
plt.savefig(RECONSTRUCTION_DIR / f"example_{random_idx}_groundtruth.png", dpi=300)
plt.show()